# CNN 전이학습 과제: 자동차 제조사(회사명) 분류

**목표:** 자동차 사진 1장에서 데이터셋의 자동차 클래스를 예측하고, 원본 폴더 라벨·대응 제조사명·Softmax 신뢰도를 출력한다.

## 프로젝트 범위 및 라벨 주의점

- 기본 구조는 `Cars/train/클래스폴더/이미지`와 `Cars/test/클래스폴더/이미지`이다. 실제 클래스 수와 이미지 수는 실행 시 결정한다.
- 기존 노트북의 폴더 확인 출력에는 `Audi`, `Hyundai Creta`, `Mahindra Scorpio`, `Rolls Royce`, `Swift`, `Tata Safari`, `Toyota Innova`가 있었다. **이 목록은 이전 출력이며, 현재 Drive 파일을 다시 확인한 결과는 아니다.**
- 제조사명과 차종명이 섞여 있다. 원본 라벨을 보존하여 학습하고 대응 제조사명은 별도 사전으로 표시한다. 이 데이터의 성능이 해당 제조사의 모든 차종에 대한 성능을 뜻하지는 않는다.
- 숫자 파일명은 식별용이며, 이미지가 들어 있는 클래스 폴더명을 정답 라벨로 사용한다.
- 모델은 ImageNet 사전학습 ResNet50이며, 기존 특징 추출층을 동결하고 마지막 `fc`만 학습한다.

## 수업 내용과 연결

수업의 `Dataset`, `DataLoader`, `weight.transforms()`, `model.fc`, `requires_grad`, `CrossEntropyLoss`, `Adam`, `classification_report`, `torch.save`, `load_state_dict`, `softmax` 흐름과 기존 셀 순서를 유지했다.
경로 정규화, 이미지 검증·중복 점검, Validation 최적 가중치 선택, 혼동행렬 분석만 과제의 실행·평가 신뢰성을 위해 보완했다. 고급 학습 라이브러리는 사용하지 않는다.

> **실행 상태:** 수정 전 출력은 마운트와 폴더 확인뿐이며 학습 로그는 없었다. 혼동을 막기 위해 이전 출력은 비웠다. 이 파일을 작성한 환경에서는 Drive 데이터·GPU 학습을 실행하지 않았다. 수치는 Colab에서 실행 후 기록한다.
> **신뢰도:** Softmax 점수는 실제 정답 확률이나 Test Accuracy를 보장하지 않는다. 학습에 없는 제조사도 기존 클래스 중 하나로 예측될 수 있다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0. 실행 전 준비

1. Google Colab에서 이 노트북을 열고 `런타임 → 런타임 유형 변경 → GPU`를 선택한다. GPU가 없으면 CPU로 동작하지만 느릴 수 있다.
2. 위 Drive 마운트 셀에서 본인 계정 연결을 완료한다.
3. 아래 `DATA_DIR`은 제공된 Cars 경로이다. 한글 NFC/NFD 차이는 경로의 각 구간을 비교해서 처리한다.
4. 위에서부터 순서대로 실행한다. 기본 설정은 배치 16, 5 Epoch이다. 새 사진 입력 셀은 파일 선택을 기다리므로 마지막에 직접 실행해도 된다.

```text
Cars/
├── train/클래스폴더/이미지
└── test/클래스폴더/이미지
```

`Train`, `Test`처럼 대소문자만 다른 이름도 확인한다. 다른 구조라면 발견한 내용을 출력하고 중단하며 원본 폴더를 이동하거나 수정하지 않는다. 아래 출력의 실제 구조를 보고 경로를 수정한다.
사전학습 가중치는 처음 모델을 불러올 때 인터넷에서 내려받는다. 최신 Colab 기본 PyTorch/torchvision을 사용하며 임의로 재설치하지 않는다.

In [ ]:
from pathlib import Path
import unicodedata

# 경로의 각 폴더를 비교하여 NFC(완성형)/NFD(분해형) 차이를 해결
def resolve_unicode_path(path):
    path = Path(path)
    if path.exists():
        return path
    current = Path(path.anchor) if path.is_absolute() else Path('.')
    parts = path.parts[1:] if path.is_absolute() else path.parts
    for part in parts:
        direct = current / part
        if direct.exists():
            current = direct
            continue
        if not current.is_dir():
            raise FileNotFoundError(f'상위 폴더가 없습니다: {current}')
        matches = [p for p in current.iterdir()
                   if unicodedata.normalize('NFC', p.name) == unicodedata.normalize('NFC', part)]
        if len(matches) != 1:
            print('확인 가능한 하위 항목:', [p.name for p in current.iterdir()][:30])
            raise FileNotFoundError(f'경로를 하나로 찾을 수 없습니다: {direct}')
        current = matches[0]
    return current

DATA_DIR = resolve_unicode_path(
    '/content/drive/MyDrive/MBCA/딥러닝 YOLO 심화과정/03 CNN 합성곱 신경망/01 Pytorch/DataSet/Cars'
)

def find_split_folder(name):
    matches = [p for p in DATA_DIR.iterdir() if p.is_dir() and p.name.casefold() == name]
    if len(matches) != 1:
        print('Cars 하위 항목:', [p.name for p in DATA_DIR.iterdir()])
        raise ValueError(f'{name} 폴더를 하나로 찾지 못했습니다. 실제 구조를 확인하세요.')
    return matches[0]

TRAIN_DIR = find_split_folder('train')
TEST_DIR = find_split_folder('test')
print('실제 데이터 폴더:', DATA_DIR)
print('Train 폴더:', TRAIN_DIR)
print('Test 폴더:', TEST_DIR)
print('Train 하위 폴더:', [p.name for p in TRAIN_DIR.iterdir() if p.is_dir()])
print('Test 하위 폴더:', [p.name for p in TEST_DIR.iterdir() if p.is_dir()])

## 1. 라이브러리 불러오기

수업에서 사용한 PyTorch와 이미지 처리 라이브러리를 불러온다.

- `torch`: 딥러닝 모델 학습
- `torchvision.models`: 사전학습 CNN 모델
- `Dataset`, `DataLoader`: 이미지와 정답 라벨을 모델에 공급
- `pandas`: 이미지 경로와 라벨을 표(DataFrame)로 관리
- `PIL`: 이미지 파일 열기
- `sklearn`: 데이터 분리 및 성능 평가

In [ ]:
import os
import random
import hashlib
import math
import time
import torchvision
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# 결과 재현을 위해 난수 고정
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# GPU 사용 가능 여부 확인
# GPU가 있으면 학습 시간을 줄일 수 있음
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('사용 장치:', device)
print('PyTorch 버전:', torch.__version__)
print('torchvision 버전:', torchvision.__version__)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU 실행: 학습이 오래 걸리면 GPU 런타임으로 변경하세요.')

## 2. 폴더에서 이미지 경로와 정답 라벨 만들기 (Q1-1 연결)

HWP에 기록된 데이터셋은 **Car Images Dataset**, 출처는 [Kaggle 원본 페이지](https://www.kaggle.com/datasets/kshitij192/cars-image-dataset)이다. 차량 정보 인식 기능의 기초를 실습하려는 선정 이유는 기존 HWP 답변을 따른다. 웹에 기재된 개수를 실제 Drive 파일 수로 대신하지 않는다.

각 클래스 폴더명을 원본 정답 라벨로 사용한다. `make_dataframe()`은 수업 2의 CSV 기반 DataFrame 대신 현재 폴더 구조에서 같은 형식의 표를 만든다.
지원 확장자를 가진 모든 파일을 읽어 디코딩 가능 여부와 SHA-256을 확인한다. 손상 파일이나 클래스 바깥 이미지가 있으면 경로를 알리고 중단하며, 임의 삭제·누락시키지 않는다. 해시는 바이트가 같은 복사본을 점검하기 위한 것으로 재인코딩·다른 각도의 유사 사진까지 판별하지는 못한다. Drive에서 최초 파일 검사에 시간이 걸릴 수 있다.

In [ ]:
image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff']

def make_dataframe(folder):
    rows = []
    bad_files = []
    loose_images = [p for p in folder.iterdir()
                    if p.is_file() and p.suffix.lower() in image_extensions]
    if loose_images:
        raise ValueError(f'클래스 폴더 밖 이미지가 있습니다. 원본 구조 확인 필요: {loose_images[:5]}')
    class_folders = sorted(p for p in folder.iterdir() if p.is_dir() and not p.name.startswith('.'))
    if not class_folders:
        raise ValueError(f'클래스 폴더가 없습니다: {folder}')
    labels = [unicodedata.normalize('NFC', p.name) for p in class_folders]
    if len(labels) != len(set(labels)):
        raise ValueError('정규화 후 같은 이름인 클래스 폴더가 둘 이상 있습니다. 원본 구조 확인 필요.')
    for class_folder, label in zip(class_folders, labels):
        image_paths = sorted(p for p in class_folder.rglob('*')
                             if p.is_file() and p.suffix.lower() in image_extensions)
        if not image_paths:
            raise ValueError(f'지원 이미지가 없는 폴더입니다: {class_folder}')
        for image_path in image_paths:
            try:
                with Image.open(image_path) as image:
                    image.verify()
                with Image.open(image_path) as image:
                    image.convert('RGB').load()
                digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
            except Exception as error:
                bad_files.append((str(image_path), str(error)))
                continue
            rows.append({'image_path': str(image_path), 'label': label, 'sha256': digest})
        print(folder.name, '/', class_folder.name, ':', len(image_paths), '장 검사')
    if bad_files:
        display(pd.DataFrame(bad_files, columns=['문제 파일', '오류']))
        raise ValueError('읽을 수 없는 이미지가 있습니다. 표시된 파일을 확인한 후 다시 실행하세요.')
    return pd.DataFrame(rows, columns=['image_path', 'label', 'sha256'])

train_df = make_dataframe(TRAIN_DIR)
test_df = make_dataframe(TEST_DIR)
if train_df.empty or test_df.empty:
    raise ValueError('Train/Test 이미지가 비어 있습니다. 경로와 폴더 구조를 확인하세요.')
raw_train_df = train_df.copy()  # 재실행 시 Train을 다시 줄이지 않도록 원본 목록 보관
print('원본 Train 이미지 수:', len(train_df))
print('원본 Test 이미지 수:', len(test_df))
display(train_df.head())

## 3. 원본 클래스 확인 및 데이터 수 분석 (Q1-2)

실제 Train 폴더에서 정렬한 클래스 목록으로 숫자 라벨을 만든다. 클래스 수를 7로 고정하지 않는다.
Test에 학습하지 않은 클래스가 있으면 평가할 수 없으므로 중단한다. Test에 일부 클래스가 없으면 경고하고 해당 클래스의 지표는 표본이 없음을 명시한다.

제조사 표시 사전은 원본 클래스와 별도로 관리한다. 알 수 없는 폴더명은 원본 라벨을 출력하며 제조사 매핑 확인이 필요하다고 알린다. `Swift`는 Suzuki 계열 차종으로 표시하고 인도 판매사 Maruti Suzuki를 병기한다. [제조사 자료](https://www.globalsuzuki.com/globalnews/2024/pdf/0509.pdf)

In [ ]:
class_names = sorted(train_df['label'].unique().tolist())
train_classes = set(class_names)
test_classes = set(test_df['label'].unique())
print('원본 자동차 클래스:', class_names)
print('실제 클래스 개수:', len(class_names))
if len(class_names) < 2:
    raise ValueError('분류 실험에는 2개 이상의 클래스가 필요합니다.')
if test_classes - train_classes:
    raise ValueError(f'Test에만 있는 클래스: {sorted(test_classes - train_classes)}')
if train_classes - test_classes:
    print('주의: Test 표본이 없는 클래스:', sorted(train_classes - test_classes))

class_to_idx = {name: i for i, name in enumerate(class_names)}
manufacturer_by_class = {
    'Audi': 'Audi',
    'Hyundai Creta': 'Hyundai',
    'Mahindra Scorpio': 'Mahindra',
    'Rolls Royce': 'Rolls-Royce',
    'Swift': 'Suzuki (인도 판매: Maruti Suzuki)',
    'Tata Safari': 'Tata Motors',
    'Toyota Innova': 'Toyota'
}
unknown_classes = [name for name in class_names if name not in manufacturer_by_class]
if unknown_classes:
    print('제조사 대응 확인 필요 (원본 라벨 보존):', unknown_classes)
display(pd.DataFrame({
    '원본 클래스': class_names,
    '제조사 표시': [manufacturer_by_class.get(name, '매핑 확인 필요') for name in class_names]
}))
print('클래스 번호 사전:', class_to_idx)

In [ ]:
# 제조사별 Train / Test 이미지 개수 집계
train_counts = train_df['label'].value_counts().reindex(class_names, fill_value=0)
test_counts = test_df['label'].value_counts().reindex(class_names, fill_value=0)

count_table = pd.DataFrame({
    'Train 이미지 수': train_counts,
    'Test 이미지 수': test_counts
})
count_table['전체 이미지 수'] = count_table['Train 이미지 수'] + count_table['Test 이미지 수']
count_table['전체 중 비율(%)'] = (
    count_table['전체 이미지 수'] / count_table['전체 이미지 수'].sum() * 100
).round(2)

display(count_table)

train_total = len(train_df)
test_total = len(test_df)
all_total = train_total + test_total

print('Train 이미지 총합:', train_total)
print('Test 이미지 총합:', test_total)
print('전체 이미지 총합:', all_total)
print(f'Train 비율: {train_total / all_total * 100:.2f}%')
print(f'Test 비율: {test_total / all_total * 100:.2f}%')
print('실제 Train:Test 비율:', f'{train_total}:{test_total}')
print('Train 제조사별 최소/최대 이미지 수:', train_counts.min(), '/', train_counts.max())

## 4. 제조사별 데이터 분포 시각화 및 샘플 확인

클래스별 이미지 수가 비슷한지 확인한다. 특정 제조사 이미지가 지나치게 많거나 적으면 학습 결과에 영향을 줄 수 있다.

아래 그림은 보고서 Q1-2에 첨부할 수 있다.

In [ ]:
# Colab 기본 글꼴에서 한글 범례가 깨지지 않도록 그래프 범례만 영어 사용
count_table[['Train 이미지 수', 'Test 이미지 수']].rename(
    columns={'Train 이미지 수': 'Train', 'Test 이미지 수': 'Test'}
).plot(kind='bar', figsize=(11, 5))
plt.title('Number of Images by Original Class')
plt.xlabel('Original class label')
plt.ylabel('Number of Images')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 각 제조사에서 샘플 이미지 1장씩 확인
fig, axes = plt.subplots(1, len(class_names), figsize=(3 * len(class_names), 3))
if len(class_names) == 1:
    axes = [axes]

for ax, class_name in zip(axes, class_names):
    sample_paths = train_df.loc[train_df['label'] == class_name, 'image_path']
    image = Image.open(sample_paths.iloc[0]).convert('RGB')
    ax.imshow(image)
    ax.set_title(class_name, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 5. Train / Validation / Test 분리 및 중복 누수 방지

제공된 Test를 최종 평가용으로 유지하고, 원본 Train에서 약 20%를 Validation으로 분리한다. 수업의 `train_test_split(..., stratify=..., random_state=42)`를 사용한다.

- 같은 파일 해시인데 라벨이 다르면 정답 충돌이므로 중단한다.
- Train에 Test와 바이트가 같은 복사본이 있으면 **그 Train 행만 학습 후보에서 제외**하고 경로를 표시한다. Test와 원본 파일은 그대로 보존한다.
- Train 내부의 동일 해시는 같은 분할에 배정하여 Train/Validation 중복을 막는다. 복사본이 많으면 실제 이미지 기준 비율은 80:20과 다를 수 있다.
- Test 내부 복사본은 개수를 알리고 제공된 평가 구성을 유지한다. 따라서 그 복사본이 지표에 반복 반영된다는 한계가 있다.
- 최종 세 분할의 경로와 해시가 겹치지 않는지 검사하고 실제 클래스별 개수·비율을 출력한다. 유사 사진이나 동일 차량의 연속 촬영 여부는 별도 육안 확인이 필요하다.

데이터 부족이나 다른 구조를 임의로 합치지 않는다. 분리가 불가능하면 이유를 출력하고 중단하며 파일을 변경하지 않는다.

In [ ]:
# 같은 내용에 다른 라벨이 붙어 있는지 먼저 점검
all_files_df = pd.concat([raw_train_df, test_df], ignore_index=True)
conflict_hashes = all_files_df.groupby('sha256')['label'].nunique()
conflict_hashes = conflict_hashes[conflict_hashes > 1].index
if len(conflict_hashes):
    display(all_files_df[all_files_df['sha256'].isin(conflict_hashes)])
    raise ValueError('동일 파일에 서로 다른 라벨이 있습니다. 정답 확인이 필요합니다.')

# Test 복사본은 학습 후보에서만 제외 (디스크의 원본 파일은 변경하지 않음)
overlap_mask = raw_train_df['sha256'].isin(set(test_df['sha256']))
excluded_train_df = raw_train_df[overlap_mask].copy()
train_pool_df = raw_train_df[~overlap_mask].copy()
print('Test와 중복되어 학습에서 제외한 원본 Train 행:', len(excluded_train_df))
if not excluded_train_df.empty:
    display(excluded_train_df[['image_path', 'label']])
print('Test 내부 동일 파일 추가 복사본 수:', int(test_df['sha256'].duplicated().sum()))

# 같은 해시를 하나의 그룹으로 보고 계층 분리한 후 모든 원본 행에 배정을 적용
unique_train_df = train_pool_df[['sha256', 'label']].drop_duplicates().reset_index(drop=True)
class_train_counts = unique_train_df['label'].value_counts().reindex(class_names, fill_value=0)
if class_train_counts.min() < 2:
    display(class_train_counts.rename('클래스별 고유 파일 수'))
    raise ValueError('Test 중복 제외 후 각 클래스에 고유 이미지 2장 이상이 필요합니다.')
val_size = max(len(class_names), math.ceil(0.2 * len(unique_train_df)))
if len(unique_train_df) - val_size < len(class_names):
    raise ValueError('계층 분리를 위한 Train/Validation 표본이 부족합니다.')
train_groups, val_groups = train_test_split(
    unique_train_df, test_size=val_size,
    stratify=unique_train_df['label'], random_state=42
)
# 매우 적은 클래스가 계층 분리의 반올림으로 빠지는 경우만 클래스별 분리
if set(train_groups['label']) != set(class_names) or set(val_groups['label']) != set(class_names):
    print('소수 클래스 보장을 위해 클래스별 80:20 분리 적용')
    train_parts, val_parts = [], []
    for class_name in class_names:
        class_groups = unique_train_df[unique_train_df['label'] == class_name]
        class_train, class_val = train_test_split(class_groups, test_size=0.2, random_state=42)
        train_parts.append(class_train)
        val_parts.append(class_val)
    train_groups = pd.concat(train_parts, ignore_index=True)
    val_groups = pd.concat(val_parts, ignore_index=True)
train_df = train_pool_df[train_pool_df['sha256'].isin(train_groups['sha256'])].reset_index(drop=True)
val_df = train_pool_df[train_pool_df['sha256'].isin(val_groups['sha256'])].reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
for column in ['image_path', 'sha256']:
    assert set(train_df[column]).isdisjoint(val_df[column]), 'Train/Validation 중복'
    assert set(train_df[column]).isdisjoint(test_df[column]), 'Train/Test 중복'
    assert set(val_df[column]).isdisjoint(test_df[column]), 'Validation/Test 중복'
assert len(train_df) + len(val_df) + len(excluded_train_df) == len(raw_train_df)
assert set(train_df['label']) == set(class_names) == set(val_df['label'])

split_table = pd.DataFrame({
    'Train': train_df['label'].value_counts(),
    'Validation': val_df['label'].value_counts(),
    'Test': test_df['label'].value_counts()
}).reindex(class_names).fillna(0).astype(int)
display(split_table)
display(train_df.groupby('label')['sha256'].nunique().reindex(class_names).rename('Train 고유 파일 수'))
print('최종 Train / Validation / Test:', len(train_df), '/', len(val_df), '/', len(test_df))
effective_total = len(train_df) + len(val_df) + len(test_df)
print('사용 이미지 기준 비율(%):',
      [round(100 * len(df) / effective_total, 2) for df in [train_df, val_df, test_df]])
print('경로·동일 파일 해시 기준 분할 간 중복 없음')
print('학습 데이터 규모 판단용: Train', len(train_df), '장 / 클래스별 최소', split_table['Train'].min(), '장')

## 6. ResNet50에 맞는 이미지 전처리

사전학습 모델은 학습할 때 사용한 방식과 맞춰 이미지 크기 조정, Tensor 변환, 정규화 등을 적용해야 한다.

`ResNet50_Weights.IMAGENET1K_V2` (현재 DEFAULT에 해당)의 `transforms()`를 사용하면 해당 사전학습 가중치에 맞는 전처리를 불러올 수 있다.

이번 기본 실험에서는 구현을 단순하게 유지하기 위해 별도의 데이터 증강은 추가하지 않고(추가 실험은 마지막 절 참고), 사전학습 모델이 권장하는 전처리를 적용한다.

In [ ]:
# ImageNet으로 사전학습된 ResNet50 가중치와 권장 전처리 객체
weight = models.ResNet50_Weights.IMAGENET1K_V2
transform = weight.transforms()

print(transform)

## 7. 자동차 이미지 Dataset 만들기

수업에서 만든 `DogDataset`과 같은 원리로 `CarDataset`을 만든다.

- `__len__`: 이미지 데이터 총 개수 반환
- `__getitem__`: 이미지 파일을 열고 전처리한 뒤 숫자 라벨과 함께 반환

이미지 파일 이름은 숫자이므로 라벨로 사용하지 않고, 앞에서 만든 `class_to_idx`를 이용해 제조사 폴더명을 숫자로 바꾼다.

In [ ]:
class CarDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # RGB로 통일하여 CNN 입력 채널 수를 3개로 맞춤
        with Image.open(row['image_path']) as opened_image:
            image = opened_image.convert('RGB')
        image = self.transform(image)

        # 제조사 폴더명을 숫자 라벨로 변환
        label = class_to_idx[row['label']]

        return image, label

train_dataset = CarDataset(train_df, transform)
val_dataset = CarDataset(val_df, transform)
test_dataset = CarDataset(test_df, transform)

print('Train 데이터셋 크기:', len(train_dataset))
print('Validation 데이터셋 크기:', len(val_dataset))
print('Test 데이터셋 크기:', len(test_dataset))

## 8. DataLoader 만들기

`DataLoader`는 이미지를 `batch_size`만큼 묶어 모델에 공급한다.

- Train은 매 Epoch마다 이미지 순서를 섞는다(`shuffle=True`).
- Validation과 Test는 평가 결과를 비교하기 쉽게 순서를 섞지 않는다.
- `num_workers=0`은 Drive 파일 읽기와 Colab에서 실행 문제가 적도록 설정한 값이다.

In [ ]:
batch_size = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print('Train 배치 수:', len(train_loader))
print('Validation 배치 수:', len(val_loader))
print('Test 배치 수:', len(test_loader))

## 9. 사전학습 ResNet50 모델 불러오기 및 마지막 분류층 교체

ResNet50은 ImageNet의 1,000개 클래스를 분류하도록 사전학습되어 있다. 이번 과제에서는 실제 폴더에서 확인한 자동차 클래스를 분류해야 하므로 마지막 `model.fc`의 출력 개수를 실제 클래스 수(`len(class_names)`)로 바꾼다.

수업에서 사용한 ResNet152와 코드 구조는 같고, 연산 부담을 줄이기 위해 ResNet50을 선택했다.

In [ ]:
# 사전학습된 ResNet50 모델 불러오기
model = models.resnet50(weights=weight)

# 마지막 분류층 확인
print('기존 FC 층:')
print(model.fc)

# 마지막 FC 층의 출력 개수를 자동차 제조사 클래스 수에 맞게 변경
model.fc = nn.Linear(model.fc.in_features, len(class_names))

assert model.fc.out_features == len(class_names) == len(class_to_idx)
print('변경된 FC 층:')
print(model.fc)

# 모델을 GPU 또는 CPU로 이동
model = model.to(device)

## 10. 전이학습 전략: 특징 추출층 동결, FC 층만 학습 (Q3-1)

과제의 네 가지 상황은 다음과 같이 비교한다.

|데이터 크기|ImageNet과의 유사도|과제 가이드의 전략|
|---|---|---|
|적음|낮음|특징 추출기 동결, 상위 층 학습과 데이터 증강|
|많음|낮음|전체 미세조정 또는 처음부터 학습|
|적음|높음|분류기 교체, 특징 추출기 동결|
|많음|높음|상위 몇 층 또는 전체 미세조정|

자동차 사진은 ImageNet과 같은 자연 이미지이며 윤곽·질감·형태 특징을 공유하므로, 의료 영상 같은 도메인보다 시각적 유사도가 높다고 판단했다. 다만 세부 제조사 차이는 일반 물체 구분보다 어려울 수 있다.
**기본 전략은 3번(데이터가 적고 유사도가 높은 상황)에 해당한다는 작업 가정**으로 FC만 학습한다. 데이터가 적다는 판단은 아직 실측 결과가 아니며, 위에서 출력한 실제 Train 총수·클래스별 고유 이미지 수를 ImageNet 규모 및 모델 크기와 비교하여 Q3-1에서 확정한다. 많고 다양한 데이터가 확인되면 4번에 해당할 수 있으며 상위 층 미세조정은 선택 실험으로 남긴다. 데이터 규모의 절대 경계값을 임의로 정하지 않는다.

`requires_grad=False`로 기존 가중치를 동결한다. 학습 중 `model.eval()`을 유지하여 동결된 BatchNorm의 이동 통계도 바뀌지 않도록 한다. FC의 역전파는 가능하다. 학습량·시간을 줄이기 위한 기본 실험이며 최적 성능을 보장하는 선택은 아니다.

In [ ]:
# 기존 모든 가중치를 고정
for param in model.parameters():
    param.requires_grad = False

# 마지막 FC 분류층만 학습 가능하도록 설정
for param in model.fc.parameters():
    param.requires_grad = True

# 실제 학습 대상 파라미터 수 확인
trainable_count = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
print('학습할 파라미터 수:', trainable_count)

# 다중분류 손실함수와 최적화 알고리즘
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

## 11. 모델 학습 및 Validation 평가 (Q3-2: 코드 및 로그 첨부)

기존 기본값인 5 Epoch로 학습한다. Validation Loss가 가장 낮은 Epoch의 가중치를 복원해 최종 모델로 사용하며 Test로 Epoch를 고르지 않는다. 한 Epoch는 Train 전체 데이터를 한 번 학습하는 단위다.

수업 노트북과 마찬가지로 기존 특징 추출층을 고정한 실험에서는 `model.eval()`을 사용한다. `eval()`은 BatchNorm/Dropout 등의 동작 모드를 평가 모드로 바꾸는 것이며, `requires_grad=True`인 FC 층의 역전파와 가중치 업데이트를 막는 명령은 아니다.

GPU 메모리 부족이 발생하면 `batch_size`를 8로 줄여 DataLoader 셀부터 다시 실행한다. 메모리 부족 후에는 모델 셀부터 재실행하여 새 학습으로 시작한다. CPU라면 학습이 오래 걸릴 수 있다. Epoch별 시간과 누적 시간을 출력하지만 전체 실행시간은 데이터 크기와 Drive 속도에 따라 달라진다.

In [ ]:
epochs = 5
best_val_loss = float('inf')
best_epoch = 0
best_state_dict = None
training_start = time.perf_counter()

history = {
    'epoch': [],
    'train_loss': [],
    'train_accuracy': [],
    'val_loss': [],
    'val_accuracy': []
}

for epoch in range(epochs):
    epoch_start = time.perf_counter()
    # 수업 실습과 같은 전이학습 방식: 고정된 특징 추출기는 평가 모드로 둠
    model.eval()

    train_loss_sum = 0
    train_correct = 0
    train_total = 0

    # -------------------- Train --------------------
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

    train_loss = train_loss_sum / train_total
    train_accuracy = train_correct / train_total

    # -------------------- Validation --------------------
    model.eval()
    val_loss_sum = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            predictions = outputs.argmax(dim=1)

            val_loss_sum += loss.item() * images.size(0)
            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_accuracy = val_correct / val_total

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        # CPU 복사로 보관하여 다음 Epoch의 업데이트와 분리
        best_state_dict = {name: value.detach().cpu().clone()
                           for name, value in model.state_dict().items()}

    history['epoch'].append(epoch + 1)
    history['train_loss'].append(train_loss)
    history['train_accuracy'].append(train_accuracy)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_accuracy)

    print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Accuracy: {train_accuracy * 100:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy * 100:.2f}% | "
        f"Epoch 시간: {time.perf_counter() - epoch_start:.1f}초 | "
        f"누적: {(time.perf_counter() - training_start) / 60:.1f}분"
    )

if best_state_dict is None:
    raise RuntimeError('정상 학습된 가중치가 없습니다. 학습 로그를 확인하세요.')
model.load_state_dict(best_state_dict)
model.eval()
print(f'최종 모델: Validation Loss 최소 Epoch {best_epoch}, Loss {best_val_loss:.4f}')
history_df = pd.DataFrame(history)
display(history_df)

## 12. 학습 과정 그래프 확인

Train Loss와 Validation Loss가 어떻게 변하는지 확인한다. Train 정확도만 계속 올라가고 Validation 정확도는 정체되거나 하락한다면 과적합 가능성을 생각할 수 있다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_df['epoch'], history_df['train_loss'], marker='o', label='Train Loss')
axes[0].plot(history_df['epoch'], history_df['val_loss'], marker='o', label='Validation Loss')
axes[0].set_title('Loss by Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history_df['epoch'], history_df['train_accuracy'] * 100, marker='o', label='Train Accuracy')
axes[1].plot(history_df['epoch'], history_df['val_accuracy'] * 100, marker='o', label='Validation Accuracy')
axes[1].set_title('Accuracy by Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 13. Test 데이터로 최종 성능 평가 (과제 Q4-1)

Test 폴더는 학습과 Validation에 사용하지 않았다. Validation Loss로 선택한 모델을 Test 데이터로 평가하여 Accuracy, Precision, Recall, F1-Score를 확인한다. Macro 평균은 클래스에 같은 비중을 주고 Weighted 평균은 Test 표본 수로 가중한다. Test 결과를 보고 기본 실험 설정을 반복 조정하면 Test에도 과적합될 수 있다.

In [ ]:
model.eval()

y_true = []
y_pred = []
test_loss_sum = 0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        predictions = outputs.argmax(dim=1)

        test_loss_sum += loss.item() * images.size(0)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        y_true.extend(labels.cpu().tolist())
        y_pred.extend(predictions.cpu().tolist())

test_loss = test_loss_sum / total
test_accuracy = accuracy_score(y_true, y_pred)

print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy * 100:.2f}%')

metric_rows = []
for average in ['macro', 'weighted']:
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(len(class_names))),
        average=average, zero_division=0
    )
    metric_rows.append({'average': average, 'Precision': precision, 'Recall': recall, 'F1-Score': f1})
metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)
print('\n원본 클래스별 성능 보고서 (support=0이면 Test 표본 없음):')
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    digits=4,
    zero_division=0
))

## 14. 혼동행렬(Confusion Matrix) 분석

혼동행렬의 행은 실제 원본 클래스, 열은 모델이 예측한 원본 클래스를 의미한다.

- 대각선 값: 올바르게 분류한 이미지 수
- 대각선 이외 값: 다른 원본 클래스로 잘못 분류한 이미지 수

어떤 원본 클래스 조합에서 오분류가 많은지 확인하고, 실제 자동차 외형이나 이미지 수 차이와 관련이 있는지 생각해 본다.

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))

display_cm = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(9, 8))
display_cm.plot(cmap='Blues', values_format='d', ax=ax, xticks_rotation=45)
plt.title('Confusion Matrix - Car Manufacturer Classification')
plt.tight_layout()
plt.show()

# 대각선을 제외한 실제 오분류 조합을 많은 순으로 출력
confusion_rows = [
    {'실제 클래스': class_names[i], '예측 클래스': class_names[j],
     '오분류 수': int(cm[i, j]),
     '해당 실제 클래스 중 비율(%)': round(100 * cm[i, j] / cm[i].sum(), 2)}
    for i in range(len(class_names)) for j in range(len(class_names))
    if i != j and cm[i, j] > 0
]
if confusion_rows:
    confusion_pairs = pd.DataFrame(confusion_rows).sort_values('오분류 수', ascending=False)
    display(confusion_pairs.head(5))
else:
    print('현재 Test에서 오분류 조합이 없습니다.')

## 15. 오분류된 이미지 확인 (Q4-1·Q4-2 고찰)

혼동행렬에서 오분류가 발견되면 실제 어떤 이미지가 잘못 분류되었는지 일부 확인할 수 있다. 오분류의 원인을 외형·배경·가림·표본 수 등과 연결하려면 실제 이미지를 관찰해야 하며 추측을 결과로 단정하지 않는다. 이 셀은 분석용이며, 학습이나 최종 평가 결과를 변경하지 않는다.

In [ ]:
# Test DataLoader는 shuffle=False이므로 test_df 순서와 예측 순서가 동일함
wrong_indices = [i for i, (true_label, pred_label) in enumerate(zip(y_true, y_pred))
                 if true_label != pred_label]

print('오분류 이미지 수:', len(wrong_indices))

show_count = min(6, len(wrong_indices))
if show_count > 0:
    fig, axes = plt.subplots(1, show_count, figsize=(4 * show_count, 4))
    if show_count == 1:
        axes = [axes]

    for ax, idx in zip(axes, wrong_indices[:show_count]):
        row = test_df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        true_name = class_names[y_true[idx]]
        pred_name = class_names[y_pred[idx]]

        ax.imshow(image)
        ax.set_title(f'True: {true_name}\nPred: {pred_name}', fontsize=9)
        ax.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print('현재 Test 데이터에서는 오분류가 없습니다.')

## 16. 최종 모델 저장하기

수업과 같이 `model_state_dict`와 `class_names`를 저장한다. 여기에 제조사 표시 사전, 전처리 가중치 버전, 선택 Epoch, 학습 설정을 함께 기록하여 복원 때 라벨 순서와 전처리가 바뀌지 않게 한다.
모델은 Cars 폴더의 `car-company-resnet50.pth`에 저장된다. 별도 실험 결과를 남기려면 경로 이름을 바꾼다.

In [ ]:
MODEL_PATH = DATA_DIR / 'car-company-resnet50.pth'

torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': class_names,
    'manufacturer_by_class': manufacturer_by_class,
    'weights_name': weight.name,
    'architecture': 'resnet50',
    'best_epoch': best_epoch,
    'best_val_loss': float(best_val_loss),
    'epochs': epochs,
    'batch_size': batch_size,
    'learning_rate': 0.001
}, MODEL_PATH)
print('모델 저장 완료:', MODEL_PATH)

## 17. 저장한 모델 불러오기 및 복원 확인

수업 3과 같이 동일 구조의 모델을 만들고 가중치를 복원한다. `weights_only=True`, `map_location='cpu'`를 명시하고 저장된 전처리 버전과 클래스 순서를 사용한다.
복원 전후 동일 Test 배치의 출력이 같은지도 확인한다. 이 확인은 추가로 모델을 고르거나 성능을 조정하는 데 사용하지 않는다.

새 런타임에서 학습 없이 복원하려면 Drive 연결·경로 셀, 라이브러리 셀을 실행한 후 `MODEL_PATH = DATA_DIR / 'car-company-resnet50.pth'`만 지정하고 아래 셀과 새 사진 예측 셀을 실행한다. 저장 셀을 다시 실행할 필요는 없다.

In [ ]:
checkpoint = torch.load(MODEL_PATH, map_location='cpu', weights_only=True)
assert checkpoint['architecture'] == 'resnet50'
loaded_class_names = checkpoint['class_names']
loaded_manufacturer_by_class = checkpoint['manufacturer_by_class']
loaded_weight = models.ResNet50_Weights[checkpoint['weights_name']]
loaded_transform = loaded_weight.transforms()

loaded_model = models.resnet50(weights=None)
loaded_model.fc = nn.Linear(loaded_model.fc.in_features, len(loaded_class_names))
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model = loaded_model.to(device)
loaded_model.eval()
assert loaded_model.fc.out_features == len(loaded_class_names)

# 전체 학습 흐름에서 실행한 경우 저장 전후 출력을 비교
if 'model' in globals() and 'test_loader' in globals():
    assert loaded_class_names == class_names
    check_images, _ = next(iter(test_loader))
    check_images = check_images.to(device)
    model.eval()
    with torch.no_grad():
        torch.testing.assert_close(model(check_images), loaded_model(check_images))
    print('저장 전후 같은 배치의 출력 일치 확인')

print('모델 복원 완료 / 선택 Epoch:', checkpoint['best_epoch'])
print('분류 가능한 원본 클래스:', loaded_class_names)

## 18. 새로운 자동차 사진 1장으로 제조사 예측하기

아래 `NEW_IMAGE_PATH`에 Train/Validation/Test에 포함되지 않은 새 사진 경로를 입력한다. `None`이면 Colab 파일 업로드 창을 연다. 업로드 창은 사용자 입력이 필요하므로 전체 실행을 자동 진행하려면 경로를 미리 지정한다.

이미지 RGB 변환 → 저장된 가중치 버전의 전처리 → `unsqueeze(0)` → 복원 모델 → `softmax` 순서로 예측한다. 원본 폴더 라벨과 대응 제조사명을 함께 표시한다.
Confidence는 학습한 클래스 사이의 Softmax 점수이며 실제 정답 확률이나 전체 Test Accuracy와 다르다. 제조사별 모든 차종을 학습한 모델이 아니며 학습하지 않은 제조사에 대한 거절 기능도 없다.

In [ ]:
# 예: NEW_IMAGE_PATH = '/content/drive/MyDrive/my_new_car.jpg'
NEW_IMAGE_PATH = None  # None이면 업로드 창, 경로를 지정하면 해당 이미지 사용

if NEW_IMAGE_PATH is None:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('새 자동차 사진을 정확히 1장 선택한 뒤 이 셀을 다시 실행하세요.')
    from io import BytesIO
    with Image.open(BytesIO(next(iter(uploaded.values())))) as opened_image:
        new_image = opened_image.convert('RGB')
else:
    NEW_IMAGE_PATH = resolve_unicode_path(NEW_IMAGE_PATH)
    if not NEW_IMAGE_PATH.is_file():
        raise FileNotFoundError(f'이미지 파일이 아닙니다: {NEW_IMAGE_PATH}')
    with Image.open(NEW_IMAGE_PATH) as opened_image:
        new_image = opened_image.convert('RGB')

display(new_image)
input_image = loaded_transform(new_image).unsqueeze(0).to(device)
loaded_model.eval()
with torch.no_grad():
    outputs = loaded_model(input_image)
    probabilities = torch.softmax(outputs, dim=1)
    confidence, predicted = probabilities.max(dim=1)

predicted_name = loaded_class_names[predicted.item()]
print('예측 원본 클래스(폴더 라벨):', predicted_name)
print('대응 자동차 제조사:', loaded_manufacturer_by_class.get(predicted_name, '매핑 확인 필요'))
print(f'예측 신뢰도(Softmax Confidence): {confidence.item() * 100:.2f}%')
print('신뢰도는 실제 정답 확률이나 Test Accuracy를 보장하지 않습니다.')

# 기존 노트북의 참고용 상위 3개 점수 출력 유지
top_k = min(3, len(loaded_class_names))
top_probabilities, top_indices = probabilities[0].topk(top_k)
print('상위 예측 결과:')
for prob, idx in zip(top_probabilities, top_indices):
    print(f'{loaded_class_names[idx.item()]}: {prob.item() * 100:.2f}%')

## 19. 과제 문항별 제출 자료 연결

|문항|답변 및 노트북 위치|실행 후 기록할 것|
|---|---|---|
|Q1-1 데이터셋·출처·이유|기존 HWP 답변, §2|Kaggle 원본과 본인 데이터가 같은지 확인|
|Q1-2 규모·클래스·분포|§2~5|원본 `count_table`, 합계·비율, 최종 `split_table`, 중복 제외 수|
|Q2-1 사전학습 모델 3개 이상 비교|기존 HWP 답변과 아래 비교표|기존 답변 유지|
|Q2-2 모델 선정 이유|기존 HWP 답변, §9|ResNet50 선정 근거|
|Q3-1 크기·유사도·전략|§10|실측 규모에 근거해 3번 작업 가정 확정 또는 수정|
|Q3-2 코드·학습 로그|§6~11, §12|FC 교체·동결 코드, 5 Epoch 로그, `history_df`, 곡선|
|Q4-1 지표·혼동행렬|§13~15|Test Loss·Accuracy, Macro/Weighted P/R/F1, 주요 오분류 조합|
|Q4-2 문제·해결 시도|§11~15, 아래 고찰 안내|실제 관찰 문제와 적용한 조치·한계|

### Q2 모델 비교 요약 (기존 HWP 답변 보조)

|모델|구조와 장점|고려할 점|
|---|---|---|
|ResNet|잔차 연결로 깊은 CNN 학습을 돕는다. 수업 코드 활용이 쉽다.|ResNet50도 CPU에서는 연산 부담이 있다.|
|VGG|작은 합성곱을 반복한 직관적인 구조이다.|일반적인 VGG 분류 모델은 파라미터·메모리 부담이 크다.|
|MobileNet|분리 합성곱 등을 이용한 경량 구조이다.|경량화와 세밀한 분류 성능의 균형을 확인해야 한다.|
|EfficientNet|깊이·너비·해상도를 함께 조정한다.|모델 규모와 입력 크기에 따라 실행 부담이 달라진다.|

수업의 ResNet152와 구현 방식이 같고 연산 부담을 줄일 수 있는 ResNet50을 선택했다. 다른 모델을 실제 비교 학습한 것은 아니므로 최고 정확도라고 주장하지 않는다.

### Q4-2 고찰 작성 방법

Train Loss와 Validation Loss의 변화 및 정확도 차이를 함께 확인한다. Train만 개선되고 Validation이 악화되는 흐름이 지속되면 과적합 가능성을 설명한다. Train 지표는 Epoch 중 가중치가 갱신되는 동안 계산하므로 최종 모델의 고정된 학습 성능과 완전히 같지는 않다. 5 Epoch만으로 결론을 단정하지 않는다.
기본 실험에 적용한 조치는 특징 추출기 동결, BatchNorm 평가 모드, 검증 최적 Epoch 복원이다. 이 조치가 실제로 성능을 개선했다는 주장은 별도 비교 실험 없이 하지 않는다. 경로 오류·메모리 부족·학습 지연은 실제 발생했을 때 원인과 시행한 해결책을 적는다.

### 선택사항: 핵심 제출 완료 후 성능 개선 실험

기본 학습 완료가 우선이다. 시간이 남으면 수업 1의 `model.layer4` 동결 해제와 작은 학습률(예: 0.0001), 또는 Train에만 수평 반전 등 증강을 **한 번에 하나씩** 비교할 수 있다. 이 경우 현재 학습의 `model.eval()`을 그대로 복사하면 BatchNorm 동작 정책이 달라지므로 학습 모드와 동결 층의 모드를 별도로 검토해야 한다. Validation으로 설정을 결정하고 최종 Test는 마지막에 평가한다. 실행하지 않은 LR 조절·Dropout·증강은 시도했다고 보고하지 않는다.

## 20. 제출 전 확인

- [ ] Drive 연결과 실제 클래스·전체 이미지 수·클래스 분포를 확인했는가?
- [ ] 원본 라벨과 대응 제조사명, 특정 차종 위주 데이터라는 한계를 설명했는가?
- [ ] 중복 제외 내역 및 최종 세 분할의 분포를 확인했는가?
- [ ] Q3-1 전략의 규모 가정을 실제 수치로 보완했는가?
- [ ] 학습 로그와 곡선, 선택 Epoch, Test 지표와 혼동행렬 출력이 있는가?
- [ ] 실제 오분류 사진과 고찰을 작성했는가?
- [ ] 모델 저장·복원 출력 일치 검사가 성공했는가?
- [ ] 새 사진 1장의 원본 라벨·제조사명·Softmax 점수를 확인했는가?
- [ ] 미작성 답변 TXT의 `[Colab 실행 후 확인 필요]`를 실제 결과로 채웠는가?
- [ ] Colab에서 **출력을 포함해 노트북을 저장/다운로드**했는가?

### API 확인 자료

- [torchvision ResNet50 가중치·전처리 공식 문서](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet50.html)
- [PyTorch torch.load 공식 문서](https://docs.pytorch.org/docs/stable/generated/torch.load.html)

현재 문서의 DEFAULT는 IMAGENET1K_V2에 해당하며, 재실행 일관성을 위해 V2를 명시했다. 저장/복원은 텐서와 기본 자료형만 포함한 체크포인트 및 `weights_only=True`를 사용한다. 실제 설치 버전과 실행 성공 여부는 Colab 출력으로 확인한다.